In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
from pathlib import Path
import pickle
from sklearn.preprocessing import normalize
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm
from collections import defaultdict
from PIL import Image
from lucent.optvis import render, param, transform, objectives
import shelve
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image
import random

import matplotlib.pyplot as plt
import numpy as np
from lucent.modelzoo import inceptionv1
from PIL import Image
from torch.nn import functional as F
from lucent.optvis import param

from olt.act import InputOutputModelSnapshot
import json

from lucent.modelzoo import inceptionv1
from olt.tfms import transform
from olt.act import InputOutputModelSnapshot
from olt.show import show_single_channel_red_green_black as S
from olt.shards import raw_iter_shards, read_image_shard


plt.style.use("dark_background")


device = "cpu"
model = inceptionv1(pretrained=True)
model = model.to(device)
model = model.eval()


FLAT_IMAGE_DIR = Path(
    "/Users/hariomnarang/Desktop/personal/hiccup-ide/olt/notebooks/this-and-prev/flat-images"
)

# Manual checking 

This section looks at the top positions manually, and analyses the activation ranges of some of the top positive input neurons.  

In [ ]:
fox_df = df[(df.cluster_label == 60) & (df.channel == 55) & (df.layer_name == "mixed4e_1x1_pre_relu_conv")].reset_index()
cat_df = df[(df.cluster_label == 61) & (df.channel == 55) & (df.layer_name == "mixed4e_1x1_pre_relu_conv")].reset_index()

len(fox_df), len(cat_df)

In [ ]:
model = model.to("cpu")
w = model.get_submodule("mixed4e_1x1_pre_relu_conv").weight[55].detach().cpu().reshape(-1)
fox_row = fox_df.iloc[0]
cat_row = cat_df.iloc[0]
fox_act = _get_ip_acts_of_row(fox_row)
cat_act = _get_ip_acts_of_row(cat_row)

fp, cp = [_norm(a * w).reshape(22,24) for a in [fox_act, cat_act]]
gfox_act = fox_act.reshape(22,24)
gcat_act = cat_act.reshape(22,24)

S([fp, cp, gfox_act, gcat_act], 10, 4, viztype="local")
plt.show()

In [ ]:
# well, the second one is not really a cat, but a panda of sorts, lol
# the first one is fox
_, ax = plt.subplots(1, 2)
ax[0].imshow(Image.open(FLAT_IMAGE_DIR / f"{fox_row.input_image_key}.jpeg"))
ax[1].imshow(Image.open(FLAT_IMAGE_DIR / f"{cat_row.input_image_key}.jpeg"))
plt.show()

## (20, 13)

This is the highest activating point for both categories. The neuron is `mixed4d_pool_reduce_pre_relu_conv:29`  


- Noise range: very dense at `[-500, 0]`
- L0: very high activating categories
    - `146`: dog like head `[0, 2000]` (and above)
    - `144`: cat like face `[0, 2000]` (and above)
- L1: `[500, 1500]`
    - `139`: bird face
    - `142`: another bird face
- L2: mid ranges `[0, 1000]`
    - `135`: car frame 

It is actually confirmed that across examples of fox and cat, this neuron is very high, and working in a polysemantic manner. 

Others seem quite close to noise. We need more evidence on whether the `L2` range is used actively.  

In [ ]:
# brightest point, 493
layer_name = "mixed4d_pool_reduce_pre_relu_conv"
channel = 29
noise = get_random_sampled_activations(model, TEST_IMAGES, layer_name, channel)
plot_activations_of_neuron(layer_name, channel, ACTS_DICT, noise)

In [ ]:
# remove brightest point
pos = 20, 13
fp[pos] = 0.
cp[pos] = 0.
gfox_act[pos] = 0.
gcat_act[pos] = 0.
    
S([fp, cp, ], 5, 2)
plt.show()
S([gfox_act, gcat_act], 5, 2)
plt.show()


## (19,21)

The next highest neuron is at (19,21). This would be neuron 477 = pool:13.  

Below we have the plots, everything other than the maxima are near noise (noise is [-500,0], this is a neuron which drives down a lot of signals).  
Two clusters look useful, `67` and `55`.  

- L0
    - `67`: animal face (single concept, its firing for both, range is 700+].
- L1
    - `55`: background: `[200,800]`


Can `55` be considered noise? We are not sure yet, the upper ranges of this category do seem to fall substantially above the noise range.   
In any case, for fox and cat, the neuron seems excited, in the sense that a very high value is being used,  here `67` is firing.  

In [ ]:
# remove brightest point
pos = 19, 21
fp[pos] = 0.
cp[pos] = 0.
gfox_act[pos] = 0.
gcat_act[pos] = 0.
    
S([fp, cp, ], 5, 2)
plt.show()
S([gfox_act, gcat_act], 5, 2)
plt.show()


In [ ]:


layer_name = "mixed4d_pool_reduce_pre_relu_conv"
channel = 13

print("value of neuron in current example", gfox_act[19, 21], gcat_act[19,21])
noise = get_random_sampled_activations(model, TEST_IMAGES, layer_name, channel)
plot_activations_of_neuron(layer_name, channel, ACTS_DICT, noise)

In [ ]:
all_layers

## (15,23)

Only active in cat, not in fox.  

- flattened nueron: 383
- `mixed4d_3x3_pre_relu_conv:271`

- noise `[-500, 0]`
- L0 `[500, 1500]`
    - `31`: white fur on animal 
    - `30`: white fur on animal
    - `28`: white fur on animal
    - In general, it seems the clusters have split up to model a monosematic function only. The pointwise multiplications of these clusters are also very similar.
- L1 `[0, 800]`
    - `27`: thin rod
    - `16`: human face
    - `22`: keyboard like
    - `29`: white dog legs
    - `24`: snake like

In [ ]:
# its fine, we can use min > as l0, and median above as l1
# instead of mean, i would use some percentile, like 90
get_labels_above_noise_range(ACTS_DICT, noise, "median").keys()


In [ ]:
sub = pdf61[(pdf61.dep_layer == "mixed4d_3x3_pre_relu_conv") & (pdf61.dep_channel == 4)]

In [ ]:
# pos = (15, 23)
# layer_name, channel = _get_layer_and_chan_from_grid_pos(*pos)

# print("value of neuron in current example", gfox_act[pos], gcat_act[pos])
layer_name = "mixed4d_3x3_pre_relu_conv"
channel = 4
noise = layer_by_channel_by_noise[layer_name][str(channel)]
plot_activations_of_neuron(layer_name, channel, ACTS_DICT, noise)

This is quite time taking. We would like to sort the points in both together, and have this go at it in one go.   
A big report is useful. For that, what would be my steps?  


First sort all the points by decreasing order for both together. As we see a point, we show stats for both.  
- We want to show the noise range of that neuron
- The activation on the first one
- The activation on the second one
- We want top clusters above noise range with their own ranges, along with their photos ideally. This is the hardest part, which im winging right now. Maybe, we can take the medians, sort, and get the place before the largest drop?
    - we define L0: maximum 5 percentile points below noise max
    - L1: 50 percentile points below noise max
- Thats it.



- Go through the points of interest (positive only right now)
- Find the L0 and L1 clusters. Get their crops. Put them along with their ranges. Put the range of Fox and Cat separately. See which ranges both fall into. This can simply be done by check which cluster ranges coincide with our points, if none, its noise.
    - We would like to show all L0 and L1 clusters as a scrollable panel tabset, easy. it would be folded in the beginning
    - For the cat and fox, we'll also show scrollable panel tabset if its range coincides with any cluster.
- done.

In [ ]:
BASE_REPORT_DIR = Path("mass-train-reports")
! ls {BASE_REPORT_DIR}

In [ ]:
get_cluster_photo(
    BASE_REPORT_DIR,
    layer_name,
    channel,
    29,
    kind="combined",
    crop_max_height=470,
)

In [ ]:
get_labels_above_noise_range(ACTS_DICT, noise, 5).keys()

get_labels_above_noise_range(ACTS_DICT, noise, 50).keys()

There are some simple things i can do now. First is checking if all the high neurons have their high activation ranges in.  

The easiest way is to sort the patch activations, plot the high act clusters, and then plot a horizontal line for the current value.   

In [ ]:
import pickle
with open("layer_by_channel_by_noise_acts.pkl", "rb") as f:
    layer_by_channel_by_noise_acts = pickle.load(f)

In [ ]:
def indices_for_percentage(arr, pct):
    arr = np.array(arr)
    shape = arr.shape
    flat = arr.ravel()

    pos_idx = np.where(flat > 0)[0]
    neg_idx = np.where(flat < 0)[0]

    def top_indices(idx, values, target_pct):
        order = idx[np.argsort(-np.abs(values[idx]))]
        total = np.abs(values[order]).sum()
        target = total * target_pct
        cumsum = np.cumsum(np.abs(values[order]))
        cutoff = min(np.searchsorted(cumsum, target) + 1, len(order))
        return order[:cutoff]

    pos_flat = top_indices(pos_idx, flat, pct)
    neg_flat = top_indices(neg_idx, flat, pct)

    pos_result = np.unravel_index(pos_flat, shape)
    neg_result = np.unravel_index(neg_flat, shape)

    return pos_result, neg_result


In [ ]:

def mean_cosine_similarity(batch_tensor, ref_tensor):
    # batch_tensor: [B, C, K, K]
    # ref_tensor: [C, K, K]
    B = batch_tensor.shape[0]
    x = batch_tensor.reshape(B, -1)        # [B, C*K*K]
    r = ref_tensor.reshape(1, -1)          # [1, C*K*K]
    sims = F.cosine_similarity(x, r, dim=1)  # [B]
    return sims.mean()

def min_euclidean_distance(batch_tensor, ref_tensor):
    # batch_tensor: [B, C, K, K]
    # ref_tensor: [C, K, K]
    B = batch_tensor.shape[0]
    x = batch_tensor.reshape(B, -1)        # [B, C*K*K]
    r = ref_tensor.reshape(1, -1)          # [1, C*K*K]
    dists = torch.norm(x - r, dim=1)       # [B]
    return dists.min()

def closest_pw(patch, w, layer_name, channel, cluster_patch_set_dir=CLUSTER_PATCH_SET_DIR):
    # get the neuron's acts first
    # dict[cid, shape[b, ip-c, k, k]]
    cid_by_patches = torch.load(CLUSTER_PATCH_SET_DIR / layer_name / str(channel) / "cid_by_patches.pt", weights_only=False)
    cid_by_sim = {}
    for cid, patches in cid_by_patches.items():
        # multiply weight for pw similarity
        # patches: [B, C, K, K]
        # patch: [C, K, K]
        # w: [C, K, K]
        cid_by_sim[cid] = mean_cosine_similarity(w*patches, w*patch)

    max_sim = 0
    best_cid = -1
    for cid, sim in cid_by_sim.items():
        if max_sim < sim:
            max_sim = sim
            best_cid = cid
    return best_cid, max_sim, cid_by_patches[best_cid]


def get_neuron_closest_cluster(model, layer_name, b, channel, y, x, captured_acts, cluster_patch_set_dir=CLUSTER_PATCH_SET_DIR):
    layer = model.get_submodule(layer_name)
    y0,y1 = _receptive_block(y, layer.kernel_size[0], layer.stride[0], layer.padding[0])
    x0,x1 = _receptive_block(x, layer.kernel_size[1], layer.stride[1], layer.padding[1])

    patch = captured_acts[layer_name]["input"][b, :, y0:y1, x0:x1] # [C, k, k]
    w = layer.weight[channel].detach().cpu().clone() # [C, k, k]
    pw = w * patch # [C, k, k]

    best_cid, max_sim, all_patches = closest_pw(patch, w, layer_name, channel, cluster_patch_set_dir)

    return best_cid, max_sim, patch, all_patches, w

In [ ]:
cur_act["mixed4d_3x3_pre_relu_conv"]["input"].shape

In [ ]:
model.get_submodule('mixed4d_3x3_pre_relu_conv').weight.shape

In [ ]:
# what is a low hanging frtuit though? feature viz for sure
from lucent.optvis import render
from olt.show import rd_bk_gn, get_local_image_limits, show_single_channel_red_green_black  as S
from torch.nn import functional as F

def _receptive_block(i, ksize, stride, padding, input_size=None):
    """
    Returns [start, end) input indices (end=exclusive) that influence
    output position i of a conv layer.
    """
    start = i * stride - padding
    end = start + ksize

    if input_size is not None:
        start = max(start, 0)
        end = min(end, input_size)

    return start, end



@objectives.wrap_objective()
def pw_across_channel_with_ksize(layer, next_neuron_weight, pws, positions, ksize, stride, padding, batch=None):
    @objectives.handle_batch(batch)
    def inner(model):
        o = model(layer)
        # mse = 0
        sim = 0
        for pw, pos in zip(pws, positions):
            # y, x = pos
            y, y1 = _receptive_block(pos[0], ksize[0], stride[0], padding[0])
            x, x1 = _receptive_block(pos[1], ksize[1], stride[1], padding[1])
            
            pw0 = o[:, :, y:y1, x:x1].reshape(-1)*next_neuron_weight
            sim += F.cosine_similarity(pw0, pw, 0)
            # mse += F.mse_loss(pw0, pw)
        return -sim
        # return mse
    return inner


def get_feature_viz_for_neuron_at(model, captured_acts, layer_name, channel, position):
    # padding is 0, since each layer has 0 padding only, and we directly look at input
    layer_by_spec = {
        "mixed4d_1x1_pre_relu_conv": ("mixed4c", (0,0)),
        "mixed4d_3x3_pre_relu_conv": ("mixed4d_3x3_bottleneck_pre_relu_conv", (1,1)),
        "mixed4d_pool_reduce_pre_relu_conv": ("mixed4c", (1,1)),
        "mixed4d_5x5_pre_relu_conv": ("mixed4d_5x5_bottleneck", (2,2)),
        "mixed4e_1x1_pre_relu_conv": ("mixed4d", (0,0)),
    }
    
    layer = model.get_submodule(layer_name)
    
    y0, y1 = _receptive_block(position[0], layer.kernel_size[0], layer.stride[0], layer.padding[0])
    x0, x1 = _receptive_block(position[1], layer.kernel_size[1], layer.stride[1], layer.padding[1])
    patch = captured_acts[layer_name]["input"][0, :, y0:y1, x0:x1]
    w = layer.weight[channel].detach().cpu()
    pw = patch * w 

    print("pw shape", pw.shape, "w", w.shape)
    
    prev_layer_name, padding = layer_by_spec[layer_name]
    svizs = render.render_vis(
        model, 
        pw_across_channel_with_ksize(prev_layer_name, w.reshape(-1), [pw.reshape(-1)], [position], layer.kernel_size, layer.stride, padding, batch=None),
        verbose=False, 
        show_image=False, 
        thresholds=(256,),
    )
    return svizs[-1][0]
    # rendered_images.append(svizs[-1][0])
    

In [ ]:
# mixed4d_3x3_pre_relu_conv:149, 13
sviz= get_feature_viz_for_neuron_at(model, cur_act, "mixed4d_3x3_pre_relu_conv", 149, [cat_row.y_position, cat_row.x_position])

In [ ]:
# we would also like to see the cluster, most similar to us :)
# for that, we find the average cosine similarity with each cluster, and use that
# i would need the input activation too. and, it is useful to see what the previous neuron is doing by using feature visualisation also
# lets do the first thing though, we get the previous layer activations full
# feature visualisation of pointwise multiplications of that neuron is also helpful (i might do cosine similarity too maybe?)
# this is now a lot of work. i need to first collect cluster by activations for each neuron
# you basically go through the full df, go through each layer-name channel, then for each, sample some number of cluster points
# then create a simple .pt file containing the pws which were clustered
# we will do l2 similarity check (or cosine). and find the cluster with max similarity, maybe single linkage.

make_report_for_neuron_act_ranges(cat_act_2, cp2, ACTS_DICT, layer_by_channel_by_noise_acts)

Now, for a given point, i would like to find the most similar cluster, for that neuron.  
We will use cosine similarity. the first think i want to do though, is check what similarities we get.  


- get the image
- get the position of interest
- get the input neuron of interest. find its own patch (by using receptive field, we would definitely need the position of interest here)
- Now my original neuron is 1x1 to itna problem nai padra. jab 3x3 hoga, to matter hoga, because multiple chize saath mei hogi.
- For now, let us not complicate it. we want to see what we are catching in our place.



for the neuron of interest, we have a position. from that we get the receptive block. each element in the receptive block corresponds to a neuron and a position in the output of that neuron, we use that to find receptive block of that neuron, then we get the patch. now for that neuron, get the clustered patches. from that find the cosine similarity.  

# Clean impl

# manual act collection

In [ ]:
from collections import defaultdict
import pickle
from tqdm import tqdm
import pandas as pd

from PIL import Image

from olt.act import InputOutputModelSnapshot
from olt.tfms import transform


def collect_acts_for_all_image(
    df, model, all_layers, device, flat_image_dir
):
    all_image_keys = df.input_image_key.unique()
    layer_by_chan_by_cluster_id_by_act = defaultdict(
        lambda: defaultdict(lambda: defaultdict(list))
    )
    for image_key in tqdm(all_image_keys):
        batch = transform(Image.open(flat_image_dir / f"{image_key}.jpeg"))[None].to(device)
        idf = df.loc[[image_key]]

        with torch.no_grad():
            acts = InputOutputModelSnapshot.get_activations(batch, model, all_layers)
    
        for tup in idf.itertuples():
            layer_by_chan_by_cluster_id_by_act[tup.layer_name][str(tup.channel)][
                str(tup.cluster_label)
            ].append(
                acts[tup.layer_name]["output"][
                    0, tup.channel, tup.y_position, tup.x_position
                ].item()
            )

    return layer_by_chan_by_cluster_id_by_act



In [ ]:
chan = 0
ndf = pd.read_csv(f"/Users/hariomnarang/Desktop/personal/hiccup-ide/olt/notebooks/this-and-prev/mass-train-reports/mixed4e_1x1_pre_relu_conv/{chan}/report.csv")
ndf = ndf[ndf.cluster_label != -1]
ndf = ndf.set_index("input_image_key", drop=False)

model = model.to("mps")
new_layer_by_chan_by_cluster_id_by_act = collect_acts_for_all_image(
    ndf, model, ["mixed4e_1x1_pre_relu_conv"], "mps", FLAT_IMAGE_DIR
)
model = model.to("cpu")

In [ ]:
from olt.act_ranges import show_output_activation_noise_histograms, show_label_points_grid
show_label_points_grid(
    new_layer_by_chan_by_cluster_id_by_act["mixed4e_1x1_pre_relu_conv"][str(chan)], 2, 3
)

In [ ]:
# mixed4e_25_df = pd.read_csv("/Users/hariomnarang/Desktop/personal/hiccup-ide/olt/notebooks/this-and-prev/mass-train-reports/mixed4e_1x1_pre_relu_conv/25/report.csv")
# mixed4e_25_df = mixed4e_25_df[mixed4e_25_df.cluster_label != -1]
# mixed4e_25_df = mixed4e_25_df.set_index("input_image_key", drop=False)


# model = model.to("mps")
# layer_by_chan_by_cluster_id_by_act_4e_12 = collect_acts_for_all_image(
#     mixed4e_25_df, model, ["mixed4e_1x1_pre_relu_conv"], "mps", FLAT_IMAGE_DIR
# )

In [ ]:
from olt.act_ranges import show_output_activation_noise_histograms, show_label_points_grid
show_label_points_grid(
    layer_by_chan_by_cluster_id_by_act_4e_12["mixed4e_1x1_pre_relu_conv"]["25"], 2, 3
)

In [ ]:
from olt.act_ranges import show_output_activation_noise_histograms, show_label_points_grid
show_label_points_grid(
    layer_by_chan_by_cluster_id_by_act_4e_12["mixed4e_1x1_pre_relu_conv"]["12"], 2, 3
)

In [ ]:
from lucent.optvis import render

svizs = render.render_vis(model.to("cpu"), "mixed4e_1x1_pre_relu_conv:212", thresholds=(256,), show_image=False)
plt.imshow(svizs[-1][0])
plt.show()

# Start report gen

In [ ]:
from pathlib import Path

import pandas as pd
from tqdm import tqdm

from olt.act_ranges import (
    BRANCHES_BY_CURRENT_LAYER,
    F_PAD_MANUAL_BY_CURRENT_LAYER,
    LAYER_NAME_BY_SHAPE,
    FeatureVizConfig,
    FlattenedChannelMap,
    NeuronParentAnalyser,
    PwSamplesConfig,
    ReportConfig,
    merge_pw_samples_into,
    print_report_for_neuron,
)

rng = np.random.default_rng(42)


def _get_stats_df(
    cluster_label,
    current_layer,
    current_channel,
    df,
    model,
    device,
    acts_dict,
    layer_by_channel_by_noise,
    cluster_patch_set_dir,
    base_report_dir,
    flat_image_dir,
    spatial_positions=None,
):
    cluster_df = df[
        (df.cluster_label == cluster_label)
        & (df.channel == current_channel)
        & (df.layer_name == current_layer)
    ].reset_index()

    analyser = NeuronParentAnalyser(
        model,
        current_layer,
        current_channel,
        list(LAYER_NAME_BY_SHAPE.keys()),
        acts_dict,
        layer_by_channel_by_noise,
        cluster_patch_set_dir,
        base_report_dir,
        flat_image_dir,
        F_PAD_MANUAL_BY_CURRENT_LAYER[current_layer],
        FlattenedChannelMap(BRANCHES_BY_CURRENT_LAYER[current_layer]),
    )

    all_stats_dfs = []
    pw_samples = {}
    for row in tqdm(
        cluster_df.itertuples(),
        total=len(cluster_df),
        desc=f"cluster {cluster_label} statistics collection",
    ):
        acts = analyser.get_activations_for_image(row.input_image_key, device)
        stats_df, pw_by_dep = analyser.collect_cluster_stats_df(
            row.y_position,
            row.x_position,
            acts,
            image_key=row.input_image_key,
            spatial_positions=spatial_positions,
        )
        stats_df["input_image_key"] = row.input_image_key
        if not stats_df.empty:
            # collect_cluster_stats_df returns a columnless empty df when a probe
            # is skipped (out-of-bounds receptive field / no cluster match). Concatenating
            # those in causes pandas to upcast int columns to float64, since it reserves
            # NaN room for any column missing from a frame even when that frame has 0 rows.
            all_stats_dfs.append(stats_df)
        merge_pw_samples_into(pw_samples, pw_by_dep)
    stats_df = pd.concat(all_stats_dfs) if all_stats_dfs else pd.DataFrame()

    return stats_df, pw_samples


def write_cluster_report(
    cluster_label,
    current_layer,
    current_channel,
    df,
    model,
    device,
    acts_dict,
    layer_by_channel_by_noise,
    cluster_patch_set_dir,
    base_report_dir,
    flat_image_dir,
    out_dir,
    concentration_metric="weighted",
    max_points_per_cluster=20,
    pw_max_samples=5,
    feature_viz_neurons=None,
    spatial_positions=None,
):
    """
    Collects stats_df for one (current_layer, current_channel, cluster_label)
    — one row of df per matching (input_image_key, y_position, x_position)
    probe — and writes its Quarto report under out_dir: out_dir/report.qmd,
    with every asset (scatter plots, cluster photos, pw-sample grids,
    concentration sparklines) written under out_dir/assets. Pure: every
    input is a param, no notebook globals touched, only writes under out_dir.

    spatial_positions: optional collection of (rel_y, rel_x) tuples, 0-indexed
    within current_layer's own receptive-field kernel (e.g. [(0, x) for x in
    range(5)] for the top row of a 5x5 kernel) — passed straight through to
    NeuronParentAnalyser.collect_cluster_stats_df to restrict this report to
    only contributions from those spatial positions. None (default) covers the
    whole kernel, same as before.
    """
    stats_df, pw_samples = _get_stats_df(
        cluster_label,
        current_layer,
        current_channel,
        df,
        model,
        device,
        acts_dict,
        layer_by_channel_by_noise,
        cluster_patch_set_dir,
        base_report_dir,
        flat_image_dir,
        spatial_positions=spatial_positions,
    )

    out_dir = Path(out_dir)
    assets_dump_dir = out_dir / "assets"
    assets_dump_dir.mkdir(parents=True, exist_ok=True)

    print_report_for_neuron(
        stats_df,
        base_report_dir=base_report_dir,
        assets_dump_dir=assets_dump_dir,
        assets_ref_dir="assets",
        origin_cluster_label=cluster_label,
        layer_by_channel_by_noise=layer_by_channel_by_noise,
        layer_by_channel_by_label_by_points=acts_dict,
        output_path=out_dir / "report.qmd",
        config=ReportConfig(
            concentration_metric=concentration_metric,
            max_points_per_cluster=max_points_per_cluster,
            sort_order="firing_frequency",
            fraction_above_threshold=0.8,
            pw_samples=PwSamplesConfig(
                layer_by_channel_by_cid_by_pw_samples=pw_samples,
                max_samples=pw_max_samples,
            ),
            feature_viz=FeatureVizConfig(model, neurons=feature_viz_neurons),
        ),
        cluster_patch_set_dir=cluster_patch_set_dir,
    )
    print("wrote report to", out_dir)
    return stats_df


# All reports

In [ ]:
from lucent.optvis import render

svizs = render.render_vis(model.to("cpu"), "mixed4e_1x1_pre_relu_conv:0", thresholds=(256,), show_image=False)
plt.imshow(svizs[-1][0])
plt.show()

## Testing last layer

In [ ]:
cluster_labels = [
    11,
    # 61,
    # 31,
    # 53, 24, 59, 45, 48, 60, 32, 46, 18, 41, 
]

In [ ]:
cluster_label = 11
stats_df, pw_samples = _get_stats_df(
    cluster_label,
    CURRENT_LAYER,
    CURRENT_CHANNEL,
    df,
    model,
    device,
    ACTS_DICT,
    layer_by_channel_by_noise,
    cluster_patch_set_dir=CLUSTER_PATCH_SET_DIR,
    base_report_dir=BASE_REPORT_DIR,
    flat_image_dir=FLAT_IMAGE_DIR,
    # spatial_positions=[(0,0)],
)


In [ ]:

show_output_activation_noise_histograms(stats_df, layer_by_channel_by_noise)

In [ ]:
def show_5x5_patch(patch, show_kwargs=None):
    if show_kwargs is None:
        show_kwargs = {}
    # shape: [48,5,5]
    arr = []
    H, W = (6,8)
    for y in range(5):
        for x in range(5):
            arr.append(patch[:, y, x])
    S([a.reshape(6,8) for a in arr], (5*3, 5*3), 5, **show_kwargs)
    plt.show()

def show_one_sample_data(
    row
):
    # row = df[(df.cluster_label == 11)].iloc[0]
    ikey = row.input_image_key
    pil = Image.open(FLAT_IMAGE_DIR / f"{ikey}.jpeg")
    batch = transform(pil)[None]
    act = InputOutputModelSnapshot.get_activations(batch, model, ["mixed5b_5x5_pre_relu_conv"])["mixed5b_5x5_pre_relu_conv"]
    
    w, ksize, stride, padding = get_layer_params(model, "mixed5b_5x5_pre_relu_conv", 12)
    y0, y1 = receptive_block(row.y_position, ksize[0], stride[0], padding[0], 11)
    x0, x1 = receptive_block(row.x_position, ksize[1], stride[1], padding[1], 11)
    
    patch = act["input"][0, :, y0:y1, x0:x1]

    hm = make_overlay_heatmap(
        model, "mixed5b_5x5_pre_relu_conv", (12, row.y_position, row.x_position), Image.open(FLAT_IMAGE_DIR / f"{ikey}.jpeg"), device="cpu"
    )
    plt.imshow(hm)
    plt.show()
    pw = patch * w
    print(pw.sum())
    print(pw.sum(dim=0))
    show_5x5_patch(pw, {"viztype": "local"})
    return patch, pw

In [ ]:
cluster_df = df[(df.cluster_label == 11) & (df.layer_name == "mixed5b_5x5_pre_relu_conv") & (df.channel == 12)]

In [ ]:
from olt.act_ranges.layer_utils import receptive_block
@objectives.wrap_objective()
def _pw_objective(
    hook_layer_name,
    prep_fn,
    dep_weight,
    pw,
    position,
    ksize,
    stride,
    padding,
    batch=None,
):
    """One render_vis call's objective: drives the image parameterization so
    that dep_layer_name's own pointwise-multiplication response at `position`
    matches `pw` (a single wild sample's dep_w * dep_patch)."""

    @objectives.handle_batch(batch)
    def inner(model):
        o = model(hook_layer_name)
        if prep_fn is not None:
            o = prep_fn(o)
        # print(position[0], ksize[0], stride[0], padding[0], o.shape[-2])
        y0, y1 = receptive_block(
            position[0], ksize[0], stride[0], padding[0], input_size=o.shape[-2]
        )
        x0, x1 = receptive_block(
            position[1], ksize[1], stride[1], padding[1], input_size=o.shape[-1]
        )
        # print(y0,y1,x0,x1)
        block = o[:1, :, y0:y1, x0:x1].reshape(-1) * dep_weight
        return F.mse_loss(block, pw)

    return inner




In [ ]:
svizs = render.render_vis(model, "mixed5b_5x5_pre_relu_conv:12")
plt.imshow(svizs[-1][0])
plt.show()

In [ ]:
high_pil = Image.fromarray(np.floor(svizs[-1][0] * 255).astype(np.uint8))
high_pil

In [ ]:
high_pil

In [ ]:
act["output"].shape

In [ ]:
batch = transform(high_pil)[None]
w = model.get_submodule("mixed5b_5x5_pre_relu_conv").weight[12].detach()
act = InputOutputModelSnapshot.get_activations(batch, model, ["mixed5b_5x5_pre_relu_conv"])["mixed5b_5x5_pre_relu_conv"]

patch = act["input"][0, :, 4:9, 4:9]
act["output"][0, 12]

In [ ]:
make_overlay_heatmap(model, "mixed5b_5x5_pre_relu_conv", (12, 4, 4), high_pil)

In [ ]:
# patch = act["input"][0, :, y0:y1, x0:x1]

# hm = make_overlay_heatmap(
#     model, "mixed5b_5x5_pre_relu_conv", (12, row.y_position, row.x_position), Image.open(FLAT_IMAGE_DIR / f"{ikey}.jpeg"), device="cpu"
# )
# plt.imshow(hm)
# plt.show()
pw = patch * w
print(pw.sum())
print(pw.sum(dim=0))
show_5x5_patch(pw.detach(), {"viztype": "global"})

In [ ]:
row.y_position, row.x_position

In [ ]:
from olt.act_ranges.feature_viz import _pad_zero

dep_weight, ksize, stride, padding = get_layer_params(model, "mixed5b_5x5_pre_relu_conv", 12)
# dep_weight = model.get_submodule("mixed5b_5x5_pre_relu_conv").weight[12].cpu().detach().reshape(-1)
obj = _pw_objective(
    "mixed5b_5x5_bottleneck_pre_relu_conv", 
    _pad_zero((2, 2, 2, 2)), 
    dep_weight.reshape(-1), 
    pw.reshape(-1), 
    (row.y_position, row.x_position),
    ksize, 
    stride, 
    padding,
)

svizs = render.render_vis(model, obj, thresholds=(256,), show_image=False)
plt.imshow(svizs[-1][0])
plt.show()

In [ ]:
noise = layer_by_channel_by_noise["mixed5b_5x5_pre_relu_conv"]["12"]
xs = range(len(noise))
plt.scatter(xs, noise)
plt.show()

In [ ]:
ACTS_DICT["mixed5b_5x5_pre_relu_conv"]

In [ ]:
non_zero = {label: act for label, act in ACTS_DICT["mixed5b_5x5_pre_relu_conv"]["12"].items() if np.median(act) > 0}

above_100 = {label: act for label, act in non_zero.items() if np.median(act) > 100}



In [ ]:
label_by_count = {label: len(act) for label, act in ACTS_DICT["mixed5b_5x5_pre_relu_conv"]["12"].items()}

In [ ]:
for l in above_100:
    print(l, label_by_count[l], np.median(above_100[l]).item())

In [ ]:
from olt.act_ranges import show_output_activation_noise_histograms, show_label_points_grid


In [ ]:
show_label_points_grid(above_100, 2, 2)

In [ ]:
stats_df.head()

In [ ]:
alias_by_pos

# Actual start

In [ ]:
NEURONS_DATA_BASE_DIR = Path("act_range_analysis_mixed4e_55")
# NEURONS_DATA_BASE_DIR = Path("act_range_analysis_mixed5b")

In [ ]:
! ls /Users/hariomnarang/Desktop/personal/hiccup-ide/olt/notebooks/this-and-prev/mass-train-reports/mixed4e_1x1_pre_relu_conv/12

In [ ]:
df = pd.read_csv(NEURONS_DATA_BASE_DIR / "main_df.csv")

extra_dfs = []
for chan in [12, 77, 25]:
    edf = pd.read_csv(
        f"/Users/hariomnarang/Desktop/personal/hiccup-ide/olt/notebooks/this-and-prev/mass-train-reports/mixed4e_1x1_pre_relu_conv/{chan}/report.csv"
    )
    extra_dfs.append(edf[edf.cluster_label != -1])
    
    
# mixed4e_12_df = pd.read_csv("/Users/hariomnarang/Desktop/personal/hiccup-ide/olt/notebooks/this-and-prev/mass-train-reports/mixed4e_1x1_pre_relu_conv/12/report.csv")
# mixed4e_77_df = pd.read_csv("/Users/hariomnarang/Desktop/personal/hiccup-ide/olt/notebooks/this-and-prev/mass-train-reports/mixed4e_1x1_pre_relu_conv/77/report.csv")
# mixed4e_25_df = pd.read_csv("/Users/hariomnarang/Desktop/personal/hiccup-ide/olt/notebooks/this-and-prev/mass-train-reports/mixed4e_1x1_pre_relu_conv/25/report.csv")

# mixed4e_12_df = mixed4e_12_df[mixed4e_12_df.cluster_label != -1]
# mixed4e_77_df = mixed4e_77_df[mixed4e_77_df.cluster_label != -1]
df = pd.concat([df] + extra_dfs)

In [ ]:
with open(NEURONS_DATA_BASE_DIR / "layer_by_channel_by_cid_by_acts.pkl", "rb") as f:
    ACTS_DICT = pickle.load(f)
with open(NEURONS_DATA_BASE_DIR /  "layer_by_channel_by_noise_acts.pkl", "rb") as f:
    layer_by_channel_by_noise = pickle.load(f)


In [ ]:
import gc



# base_report_dir = Path("mixed5b-mass-trian-reports")
base_report_dir = Path("mass-train-reports")
BASE_REPORT_DIR = base_report_dir


CLUSTER_PATCH_SET_DIR = NEURONS_DATA_BASE_DIR / "cluster-patches-set"

# CURRENT_LAYER = "mixed5b_5x5_pre_relu_conv"
# CURRENT_CHANNEL = 12

CURRENT_LAYER = "mixed4e_1x1_pre_relu_conv"
CURRENT_CHANNEL = 77
ROOT_OUT_DIR = Path("cluster_reports")
cluster_labels = [22]
# cluster_labels = [2940]

In [ ]:
model = model.to("cpu")

In [ ]:
c61_fvis_neurons = [
    ("mixed4d_5x5_pre_relu_conv", 1),
    ("mixed4d_3x3_pre_relu_conv", 31),
    ("mixed4d_5x5_pre_relu_conv", 60),
    ("mixed4d_5x5_pre_relu_conv", 61),
    ("mixed4d_5x5_pre_relu_conv", 22),
    ("mixed4d_3x3_pre_relu_conv", 134),
    ("mixed4d_3x3_pre_relu_conv",251),
    ("mixed4d_3x3_pre_relu_conv",204),
    ("mixed4d_5x5_pre_relu_conv", 8),
    ("mixed4d_3x3_pre_relu_conv", 68),
    ("mixed4d_pool_reduce_pre_relu_conv", 24),
    ("mixed4d_3x3_pre_relu_conv", 22),
    ("mixed4d_3x3_pre_relu_conv", 258),
    ("mixed4d_1x1_pre_relu_conv", 22),
    ("mixed4d_pool_reduce_pre_relu_conv",29),

]

c77_22_fvis_neurons = [
    ("mixed4d_1x1_pre_relu_conv", 14)
]


spatial_alias = None
spatial_positions = None

for cluster_label in cluster_labels:
    
    fvis_neurons = c61_fvis_neurons if (cluster_label == 61 and CURRENT_LAYER == "mixed4e_1x1_pre_relu_conv" and CURRENT_CHANNEL == 55) else []

    fvis_neurons = c77_22_fvis_neurons if (cluster_label == 22 and CURRENT_LAYER == "mixed4e_1x1_pre_relu_conv" and CURRENT_CHANNEL == 77) else []
    
    if not (BASE_REPORT_DIR / CURRENT_LAYER / str(CURRENT_CHANNEL) / f"cluster_{cluster_label}_combined.jpeg").exists():
        print(f"Cluster: {cluster_label} does not exist, skipping")
    out_dir = ROOT_OUT_DIR / CURRENT_LAYER / str(CURRENT_CHANNEL) / str(cluster_label)
    if spatial_alias is not None:
        out_dir = out_dir / spatial_alias
    out_dir.mkdir(parents=True, exist_ok=True)
    
    write_cluster_report(
        cluster_label=cluster_label,
        current_layer=CURRENT_LAYER,
        current_channel=CURRENT_CHANNEL,
        df=df,
        model=model,
        device=device,
        acts_dict=ACTS_DICT,
        layer_by_channel_by_noise=layer_by_channel_by_noise,
        cluster_patch_set_dir=CLUSTER_PATCH_SET_DIR,
        base_report_dir=BASE_REPORT_DIR,
        flat_image_dir=FLAT_IMAGE_DIR,
        out_dir=out_dir,
        feature_viz_neurons=fvis_neurons,
        spatial_positions=spatial_positions,
    )
    gc.collect()


In [ ]:
! ls mass-trian-reports/mixed4e_1x1_pre_relu_conv/77

In [ ]:
out_dir

In [ ]:
! ls mixed5b-mass-trian-reports/mixed5b_5x5_pre_relu_conv/

In [ ]:
# from olt.act_ranges import NeuronParentAnalyser, LAYER_NAME_BY_SHAPE, MIXED4D_BRANCHES, F_PAD_MANUAL_BY_CURRENT_LAYER, FlattenedChannelMap

# CURRENT_LAYER = "mixed4e_1x1_pre_relu_conv"
# CURRENT_CHANNEL = 55

# if row.layer_name != CURRENT_LAYER or row.channel != CURRENT_CHANNEL:
#     raise Exception(f"row_layer={layer_name} current_layer={CURRENT_LAYER} row_channel={row.channel} current_channel={CURRENT_CHANNEL} are different")

# analyser = NeuronParentAnalyser(
#   model,
#   CURRENT_LAYER,
#   CURRENT_CHANNEL,
#   list(LAYER_NAME_BY_SHAPE.keys()),
#   ACTS_DICT,
#   layer_by_channel_by_noise,
#   CLUSTER_PATCH_SET_DIR,
#   BASE_REPORT_DIR,
#   FLAT_IMAGE_DIR,
#   F_PAD_MANUAL_BY_CURRENT_LAYER[CURRENT_LAYER],
#   FlattenedChannelMap(MIXED4D_BRANCHES),
# )

# acts = analyser.get_activations_for_image(row.input_image_key)
# analyser.plot_clusters(row.y_position, row.x_position, acts, lambda *a, **kw: True, stuff_to_show=["heatmap", "act_range"])

Instead of a lot of visual analysis, i now want these stats in a CSV file.  

- similarity to closest cluster
- distance to noise (wrt noise range)
- ratio of points above noise in the closest cluster
- kind (positive/negative)
- layer name
- channel
- cluster id

In [ ]:
cluster_df.head()

In [ ]:
mixed4e_55_noise = layer_by_channel_by_noise["mixed4e_1x1_pre_relu_conv"]['55']
mixed4e_55_noise_med = np.median(mixed4e_55_noise)
_, mixed4e_55_noise_max = get_noise_range(mixed4e_55_noise)
mixed4e_55_radius = mixed4e_55_noise_max - mixed4e_55_noise_med

letters_acts = np.array(ACTS_DICT["mixed4e_1x1_pre_relu_conv"]['55']['24'])
letters_dists = (letters_acts - mixed4e_55_noise_med) / mixed4e_55_radius


mixed4e_55_radius

In [ ]:
plt.hist(letters_dists)
plt.show()

In [ ]:
np.letters_acts

In [ ]:
mixed4e_55_noise_med, mixed4e_55_noise_max

In [ ]:
ys = layer_by_channel_by_noise["mixed4e_1x1_pre_relu_conv"]['55']
xs = range(len(ys))

plt.scatter(xs, ys)
plt.show()

In [ ]:
pos_merged_dfs = {}
neg_merged_dfs = {}

In [ ]:
LABEL = 61
cluster_df = df[(df.cluster_label == LABEL) & (df.channel == 55) & (df.layer_name == "mixed4e_1x1_pre_relu_conv")].reset_index()
cluster_df.head()


In [ ]:
w = model.get_submodule(row.layer_name).weight[55].cpu().detach().numpy().reshape(-1)

In [ ]:
row = cluster_df.iloc[26]
timg = transform(Image.open(FLAT_IMAGE_DIR / f"{row.input_image_key}.jpeg"))[None]
act = InputOutputModelSnapshot.get_activations(timg, model, [row.layer_name])[row.layer_name]["input"]
pw = w * act[0, :, row.y_position, row.x_position].numpy()

In [ ]:
sm_pw = pw.copy()
sm_pw[493] = 0.

In [ ]:
all_layers

In [ ]:
S([pw.reshape(22,24), sm_pw.reshape(22,24)], 10, viztype="local", ax_titles=["raw", "without the bright spot at (20,13)"])
plt.show()

In [ ]:
analyser = NeuronParentAnalyser(model, all_layers, ACTS_DICT, layer_by_channel_by_noise, CLUSTER_PATCH_SET_DIR, BASE_REPORT_DIR, FLAT_IMAGE_DIR, (0,0))

acts = analyser.get_activations_for_image(row.input_image_key)
analyser.plot_clusters(row.layer_name, row.channel, row.y_position, row.x_position, acts, lambda *a,**kw: True, stuff_to_show=["heatmap", "act_range"])

In [ ]:
analyser = NeuronParentAnalyser(model, all_layers, ACTS_DICT, layer_by_channel_by_noise, CLUSTER_PATCH_SET_DIR, BASE_REPORT_DIR, FLAT_IMAGE_DIR, (0,0))

all_stats = []


for row in tqdm(cluster_df.itertuples(), total=len(cluster_df), desc="positive"):
    current_act = analyser.get_activations_for_image(row.input_image_key)
    stats_df = analyser.collect_cluster_stats_df(
        row.layer_name, row.channel, row.y_position, row.x_position,
        current_act, kind="positive",
    )
    stats_df["input_image_key"] = row.input_image_key
    all_stats.append(stats_df)

merged_df = pd.concat(all_stats, ignore_index=True)
pos_merged_dfs[LABEL] = merged_df

all_stats = []
for row in tqdm(cluster_df.itertuples(), total=len(cluster_df), desc="negative"):
    current_act = analyser.get_activations_for_image(row.input_image_key)
    stats_df = analyser.collect_cluster_stats_df(
        row.layer_name, row.channel, row.y_position, row.x_position,
        current_act, kind="negative",
    )
    stats_df["input_image_key"] = row.input_image_key
    all_stats.append(stats_df)

merged_df = pd.concat(all_stats, ignore_index=True)
neg_merged_dfs[LABEL] = merged_df

In [ ]:
pos_merged_dfs[LABEL].head()

In [ ]:
pos_merged_dfs.keys()

In [ ]:
pdf61 = pos_merged_dfs[61]

I want to check out the points between 1-2 with high similarity (above 0.8). 
We want to look at the cluster distribution (start by median only, approximately the expected distance of this cluster from the noise point)  


So, in the end, for a single example, we want to see the median cluster distance distribution for high similarity points.  
Maybe also put a plot of the cumsum (or at least, the contribution to the cum sum)?  

In [ ]:
ikey0 = pdf61.iloc[0].input_image_key

kdf0 = pdf61[
    (pdf61.input_image_key == ikey0) &
    (pdf61.similarity > 0.7)
]
len(kdf0)

In [ ]:
pdf61

In [ ]:
sub = pdf61[(pdf61.dep_layer == "mixed4d_pool_reduce_pre_relu_conv") & (pdf61.dep_channel == 29)]

print("noise dist, min mean max", sub.noise_distance.min(), sub.noise_distance.mean(), sub.noise_distance.max())
sub.groupby("dep_cid")["best_cluster_med"].agg(count="count", mean="mean").sort_values("count", ascending=False)

In [ ]:
# now lets analyse one of the input neurons
# mixed4d_pool_reduce_pre_relu_conv:29 is the easiest one


kdf0

In [ ]:
sub = pdf61[(pdf61.dep_layer == "mixed4d_3x3_pre_relu_conv") & (pdf61.dep_channel == 4)]
print("noise dist, min mean max", sub.noise_distance.min(), sub.noise_distance.mean(), sub.noise_distance.max())
sub.groupby("dep_cid")["best_cluster_med"].agg(count="count", mean="mean").sort_values("count", ascending=False)

In [ ]:
noise_min, noise_max = get_noise_range(layer_by_channel_by_noise["mixed4e_1x1_pre_relu_conv"]["55"])
noise_med = np.median(layer_by_channel_by_noise["mixed4e_1x1_pre_relu_conv"]["55"])

In [ ]:
(np.median(ACTS_DICT["mixed4e_1x1_pre_relu_conv"]["55"]["24"]) - noise_med) / (noise_max - noise_med)